Data Preprocessing 04 - Text Data - Pure Python Pipeline

> **MLCourse · Data Science Foundations · 05_data_preprocessing**

Welcome back! So far every dataset we cleaned was a tidy grid of numbers and categories.
Text is different: it is messy *by design*, because humans - not databases - write it.
Reviews shout in CAPS, carry leftover `<br />` HTML tags, hide URLs, and say "don't" in
forty different ways. Before any machine-learning model can read text, someone has to
tame it into numbers.

In this notebook we build that entire taming pipeline **from scratch, using nothing but
the Python standard library** (`re`, `string`, `collections`) plus pandas for display.
Why from scratch? Because when you hand-roll lowercase/strip/tokenize yourself, nothing
stays magic: you see exactly what industry toolkits later automate for you. The next
notebook (05) rebuilds the same pipeline with NLTK so you can compare.

## What you'll learn

- The core vocabulary of text mining: **corpus**, **document**, **token**, **vocabulary**
- A reusable cleaning pipeline: HTML removal, URL removal, contraction expansion,
  punctuation handling (two ways), whitespace collapsing, optional digit removal
- Why naive `.split()` tokenization fails and how regex tokenizers differ
- Building and applying a stopword set - including domain-specific ones
- A naive suffix-stripping stemmer written by hand, and why it fails
- What lemmatization is and how it differs from stemming
- Bag-of-Words vectors built with `Counter`, plus a sparsity reality check
- N-grams (bigrams) from scratch - and how they rescue negations like "not good"
- TF-IDF computed with plain loops, verified against scikit-learn

In [1]:
import math
import re
import string
from collections import Counter

import numpy as np
import pandas as pd

print("Ready. Python is all we need for now.")

Ready. Python is all we need for now.


## 1. The vocabulary of text mining

Four terms come up in every NLP conversation, so let's pin them down before writing code:

| Term | Meaning | Example |
|---|---|---|
| **Corpus** | Your whole collection of texts | All 10,000 product reviews |
| **Document** | One item inside the corpus | A single review |
| **Token** | One meaningful unit (usually a word) after splitting | `"battery"`, `"great"` |
| **Vocabulary** | The set of *unique* tokens across the corpus | `{great, battery, slow, ...}` |

Keep these straight and every later concept clicks into place: we clean documents,
tokenize them into token sequences, collect the corpus-level vocabulary, and finally turn
each document into a vector indexed by that vocabulary.

In [2]:
# A tiny demonstration of the four terms on three one-line "documents".
mini_corpus = [
    "Battery life is great.",
    "The screen is great but the battery is weak.",
    "Weak speaker, great price.",
]

tokens_per_doc = [doc.lower().split() for doc in mini_corpus]
vocabulary = sorted({token for tokens in tokens_per_doc for token in tokens})

print(f"Documents in corpus : {len(mini_corpus)}")
print(f"Tokens per doc      : {[len(t) for t in tokens_per_doc]}")
print(f"Vocabulary size     : {len(vocabulary)}")
print(f"Vocabulary          : {vocabulary}")

Documents in corpus : 3
Tokens per doc      : [4, 9, 4]
Vocabulary size     : 12
Vocabulary          : ['battery', 'but', 'great', 'great.', 'is', 'life', 'price.', 'screen', 'speaker,', 'the', 'weak', 'weak.']


## 2. A deliberately messy review corpus

Real text never arrives clean. To practice honestly, we build a **tiny but realistic**
corpus of ten short app/service reviews stuffed with classic mess:

- leftover HTML tags (`<br />`)
- pasted URLs
- SHOUTING IN CAPS
- punctuation storms (`!!!`, `...`)
- contractions (`don't`, `can't`, `it's`, `wouldn't`)
- numbers and prices (`$9.99`, `45 minutes`)
- stray double spaces

Ten documents is small enough to eyeball every intermediate result - exactly what you
should do with any new text dataset before scaling up.

In [3]:
RAW_REVIEWS = [
    "This app is GREAT!!! Works perfectly, love the new design. <br /> Five stars!",
    "Terrible experience...  it crashed TWICE on launch. Don't waste your money.",
    "Not good at all. Battery drains fast and support ignored my 3 tickets.",
    "Honestly? Pretty decent for $9.99/month. Can't complain much.",
    "I was on hold for 45 minutes!!! Worst support ever. See http://complaints.example.com",
    "AMAZING update!!!  Everything loads instantly now. Well done team :)",
    "Meh. Does the job but the UI feels dated and it's slow on older phones.",
    "DO NOT subscribe. They charged me twice and wouldn't refund. Scam alert!!",
    "Good value, easy setup. Took 10 minutes and everything worked. Recommended.",
    "It's okay... sometimes great, sometimes awful. Very inconsistent wifi calling.",
]

print(f"Reviews in corpus: {len(RAW_REVIEWS)}")
print()
for i, review in enumerate(RAW_REVIEWS[:3], start=1):
    print(f"[{i}] {review!r}")

Reviews in corpus: 10

[1] 'This app is GREAT!!! Works perfectly, love the new design. <br /> Five stars!'
[2] "Terrible experience...  it crashed TWICE on launch. Don't waste your money."
[3] 'Not good at all. Battery drains fast and support ignored my 3 tickets.'


## 3. The cleaning pipeline, one brick at a time

Cleaning converts raw strings into normalized text that downstream steps can trust.
We will build it as **small, single-purpose functions** and then compose them in order:

> lowercase → strip HTML → remove URLs → expand contractions →
> (optional) drop digits → remove punctuation → collapse whitespace

Order matters! For example, we must expand contractions **before** removing punctuation,
because the apostrophe in `don't` is punctuation - strip it first and you get `dont`,
which no contraction dictionary can recognize anymore.

### 3.1 Lowercase everything

**Why:** to a computer, `"GREAT"`, `"Great"`, and `"great"` are three different tokens.
Models count words; unless case carries real meaning (named entities, code), collapsing
case triples your apparent vocabulary for zero benefit. Lowercasing first also lets our
contraction dictionary use lowercase keys only.

In [4]:
def lowercase(text):
    """Collapse all casing to lower-case."""
    return text.lower()


demo = "This App Is GREAT!!!"
print(lowercase(demo))

this app is great!!!


### 3.2 Strip HTML tags and URLs

**Why:** reviews scraped from web pages often smuggle markup like `<br />` along, and
users paste links. Neither helps a sentiment model - they are noise tokens waiting to
pollute our vocabulary. The regex `<[^>]+>` reads as: an opening `<`, then *anything
that is not a closing bracket*, then `>`. URL patterns cover `http://...` and `www....`.

In [5]:
def strip_html(text):
    """Replace anything that looks like an HTML tag with a space."""
    return re.sub(r"<[^>]+>", " ", text)


def remove_urls(text):
    """Replace http(s)://... or www.... links with a space."""
    return re.sub(r"https?://\S+|www\.\S+", " ", text)


demo = "Love it <br /> details at https://example.com/deal?ref=x"
print(strip_html(demo))
print(remove_urls(strip_html(demo)))

Love it   details at https://example.com/deal?ref=x
Love it   details at  


### 3.3 Expand contractions

**Why:** `don't` and `do not` mean the same thing but look completely different to a
tokenizer. Expanding contractions merges those counts. We keep the mapping small and
honest - a real project would ship a few dozen more entries.

> ⚠️ **Common pitfall:** expanding contractions *after* punctuation removal silently
does nothing, because the apostrophe is already gone (`dont` no longer matches).
Expansion must run while apostrophes still exist.

In [6]:
CONTRACTIONS = {
    "don't": "do not",
    "can't": "can not",
    "won't": "will not",
    "isn't": "is not",
    "aren't": "are not",
    "wasn't": "was not",
    "didn't": "did not",
    "doesn't": "does not",
    "wouldn't": "would not",
    "couldn't": "could not",
    "shouldn't": "should not",
    "hasn't": "has not",
    "it's": "it is",
    "that's": "that is",
    "there's": "there is",
    "i'm": "i am",
    "i've": "i have",
    "you're": "you are",
}


def expand_contractions(text):
    """Replace contraction keys with their expanded forms (word-boundary safe)."""
    for contraction, full_form in CONTRACTIONS.items():
        pattern = r"\b" + re.escape(contraction) + r"\b"
        text = re.sub(pattern, full_form, text)
    return text


print(expand_contractions("don't worry, it's fine, i'm sure"))

do not worry, it is fine, i am sure


### 3.4 Remove punctuation - two ways compared

There are two idiomatic ways to strip punctuation, and they behave differently:

1. **Regex substitution** `re.sub(r"[^\w\s]", " ", text)` - replaces each punctuation
   character with a **space**, so `well-known` becomes `well known` (two words) and
   `$1,299` becomes `1 299`.
2. **Translation table** `text.translate(str.maketrans("", "", string.punctuation))` -
   *deletes* characters outright, so `well-known` fuses into `wellknown`.

Deletion glues words together (usually bad); substitution keeps word boundaries but can
split hyphenated compounds. We prefer substitution for general text - fused garbage
tokens are worse than split ones.

In [7]:
def remove_punct_re(text):
    """Swap every non-word, non-space character for a space (keeps boundaries)."""
    return re.sub(r"[^\w\s]", " ", text)


def remove_punct_translate(text):
    """Delete punctuation outright via a translation table (fuses hyphenated words)."""
    return text.translate(str.maketrans("", "", string.punctuation))


demo = "well-known brands don't cost $1,299.00!"
print("regex     :", repr(remove_punct_re(demo)))
print("translate :", repr(remove_punct_translate(demo)))

regex     : 'well known brands don t cost  1 299 00 '
translate : 'wellknown brands dont cost 129900'


### 3.5 Collapse whitespace

**Why:** every replacement above inserts spaces, so strings sprout double/triple spaces.
`\s+` matches any run of whitespace (spaces, tabs, newlines) and squashes it to one
space; a final `.strip()` trims the edges. Always collapse *last*, right before
tokenizing.

In [8]:
def collapse_spaces(text):
    """Squash any whitespace run to a single space and trim the ends."""
    return re.sub(r"\s+", " ", text).strip()


print(repr(collapse_spaces("too   many      spaces\tand\nnewlines")))

'too many spaces and newlines'


### 3.6 Digits: keep or drop?

**Why it is a *decision* and not a rule:** digits are meaningful in some tasks and noise
in others.

- Predicting **sentiment** ("great app!!", "terrible support") → digits rarely matter;
  dropping them shrinks the vocabulary.
- Extracting **prices** or **durations** ("only $9.99", "crashed 3 times") → digits are
  literally the signal; keep them!

Our `clean_text` therefore takes a `remove_digits` flag instead of deciding for you.

### 3.7 Compose the full pipeline

Now we snap the bricks together in the order we argued for: lowercase → HTML → URLs →
contractions → (optional digits) → punctuation → whitespace. Small functions + one
composer means every step stays testable and swappable - the same architecture the big
libraries use internally.

In [9]:
def clean_text(text, remove_digits=False):
    """Full stdlib cleaning pipeline for one document."""
    text = lowercase(text)
    text = strip_html(text)
    text = remove_urls(text)
    text = expand_contractions(text)
    if remove_digits:
        text = re.sub(r"\d+", " ", text)  # digit runs become spaces, then get collapsed
    text = remove_punct_re(text)
    return collapse_spaces(text)


for raw in RAW_REVIEWS[:4]:
    print(f"RAW : {raw}")
    print(f"CLEAN: {clean_text(raw, remove_digits=True)}")
    print()

RAW : This app is GREAT!!! Works perfectly, love the new design. <br /> Five stars!
CLEAN: this app is great works perfectly love the new design five stars

RAW : Terrible experience...  it crashed TWICE on launch. Don't waste your money.
CLEAN: terrible experience it crashed twice on launch do not waste your money

RAW : Not good at all. Battery drains fast and support ignored my 3 tickets.
CLEAN: not good at all battery drains fast and support ignored my tickets

RAW : Honestly? Pretty decent for $9.99/month. Can't complain much.
CLEAN: honestly pretty decent for month can not complain much



> 💡 **Pro tip:** keep the raw column untouched and store cleaned text in a *new*
column. Cleaning is destructive and opinionated; the day your task changes (e.g., you
suddenly need prices), the original text is irreplaceable.

## 4. Tokenization - cutting text into units

A **tokenizer** decides where one token ends and the next begins. It sounds trivial -
until it isn't. Let's watch naive approaches stumble on one nasty sentence containing a
hyphenated word, a contraction, a year, and an acronym: `"well-known it's 2024 U.S.A."`

### 4.1 Failure demo: naive `.split()`

`.split()` only breaks on whitespace. Punctuation stays glued to words (`U.S.A.`),
punctuation-only fragments survive alone (`.`), and every variant (`great!`,
`great`, `great!!!`) becomes a *different* vocabulary entry.

In [10]:
tricky = "well-known it's 2024 U.S.A."

naive_tokens = tricky.split()
print(naive_tokens)

['well-known', "it's", '2024', 'U.S.A.']


### 4.2 Two regex tokenizers and their trade-offs

With `re.findall` we choose *what counts as a token* via a character class:

- `[a-z']+` on lowercased text - letters plus apostrophes: keeps contractions whole
  (`it's`) but drops digits entirely and shreds the acronym (`u.s.a.` → `u`,`s`,`a`).
- `\b\w+\b` - letters/digits/underscore runs: keeps `2024` but splits contractions
  (`it's` → `it`, `s`).

Neither wins universally. The lesson: **the tokenizer encodes assumptions about what
information matters** - pick it deliberately, and note it down like any hyperparameter.

In [11]:
lowered = tricky.lower()

pattern_alpha_apos = re.findall(r"[a-z']+", lowered)
pattern_word_chars = re.findall(r"\b\w+\b", lowered)

print(f"[a-z']+   -> {pattern_alpha_apos}")
print(r"\b\w+\b   ->", pattern_word_chars)

[a-z']+   -> ['well', 'known', "it's", 'u', 's', 'a']
\b\w+\b   -> ['well', 'known', 'it', 's', '2024', 'u', 's', 'a']


> 💡 **Pro tip:** after our §3 cleaning (punctuation already removed), plain
`[a-z]+` is equivalent to splitting on whitespace - simple *and* predictable. That is
what we'll use below; notebook 05 swaps in NLTK's statistically trained tokenizer.

In [12]:
TOKEN_RE = re.compile(r"[a-z]+")


def tokenize(text):
    """Extract alphabetic tokens from already-cleaned text."""
    return TOKEN_RE.findall(text)


cleaned_reviews = [clean_text(raw, remove_digits=True) for raw in RAW_REVIEWS]
tokenized_reviews = [tokenize(doc) for doc in cleaned_reviews]

for doc, tokens in zip(cleaned_reviews[:2], tokenized_reviews[:2]):
    print(f"{doc}\n  -> {tokens}\n")

this app is great works perfectly love the new design five stars
  -> ['this', 'app', 'is', 'great', 'works', 'perfectly', 'love', 'the', 'new', 'design', 'five', 'stars']

terrible experience it crashed twice on launch do not waste your money
  -> ['terrible', 'experience', 'it', 'crashed', 'twice', 'on', 'launch', 'do', 'not', 'waste', 'your', 'money']



## 5. Stopwords - the filler words

**Stopwords** are ultra-frequent function words (`the`, `is`, `to`) that appear in almost
every document. For tasks like classification they add frequency noise without
discriminative power, so we often filter them. Below is a hand-built ~40-word English
starter set - notice what is **missing**: `not`, `no`, `never`. More on that in a moment.

In [13]:
STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "so", "because",
    "i", "me", "my", "we", "our", "you", "your", "he", "she", "it",
    "they", "them", "this", "that", "these", "those",
    "is", "am", "are", "was", "were", "be", "been", "being",
    "do", "does", "did", "have", "has", "had",
    "will", "would", "can", "could", "should",
    "to", "of", "in", "on", "for", "with", "at", "by", "from", "as",
}
STOPWORDS = frozenset(STOPWORDS)  # frozenset: immutable + O(1) membership tests
print(f"Hand-built stopword set size: {len(STOPWORDS)}")

sample_tokens = tokenized_reviews[1]
kept = [t for t in sample_tokens if t not in STOPWORDS]
dropped = [t for t in sample_tokens if t in STOPWORDS]
print(f"Kept   : {kept}")
print(f"Dropped: {dropped}")

Hand-built stopword set size: 55
Kept   : ['terrible', 'experience', 'crashed', 'twice', 'launch', 'not', 'waste', 'money']
Dropped: ['it', 'on', 'do', 'your']


> ⚠️ **Common pitfall:** generic stopword lists include negations (`not`, `no`,
> `never`). Filtering them turns *"not good"* into *"good"* - flipping sentiment with
> one careless line of code. For sentiment work, always audit the stopword list.

### Domain-specific stopwords

Every domain has its own filler. In app reviews, `app`, `phone`, and `update` appear in
nearly every document regardless of sentiment, so they carry little signal *here*. But
be careful: in a churn-analysis corpus, mentions like `cancel` may be exactly the signal
you need. Add domain stopwords only after eyeballing the top-frequency list.

In [14]:
filtered_reviews = [[t for t in toks if t not in STOPWORDS] for toks in tokenized_reviews]
before = sum(len(toks) for toks in tokenized_reviews)
after = sum(len(toks) for toks in filtered_reviews)
print(f"Tokens before filtering: {before}")
print(f"Tokens after filtering : {after} ({before - after} removed)")

Tokens before filtering: 114
Tokens after filtering : 82 (32 removed)


## 6. A naive stemmer from scratch

**Stemming** chops word endings so variants collapse: *running/runs/runned* → *run*-ish.
Classic algorithms (Porter, Snowball) apply cascades of clever suffix rules. Let's build
the crudest possible version - chop common suffixes with minimum-length guards so short
words like `is` don't lose their `s` - and then *honestly* watch it fail.

In [15]:
def naive_stem(word):
    """Chop common English suffixes with min-length guards. Educational, NOT production."""
    if len(word) > 5 and word.endswith("ing"):
        return word[:-3]
    if len(word) > 4 and word.endswith("ied"):          # tried -> try
        return word[:-3] + "y"
    if len(word) > 4 and word.endswith("ed"):
        return word[:-2]
    if len(word) > 4 and word.endswith("es"):           # studies -> studi (oops!)
        return word[:-2]
    if len(word) > 4 and word.endswith("ly"):
        return word[:-2]
    if len(word) > 3 and word.endswith("s"):
        return word[:-1]
    return word


test_words = [
    "running", "studies", "flies", "happily", "universities",
    "wanted", "jumps", "cats", "better", "was", "good",
]
stem_results = pd.DataFrame(
    {"word": test_words, "naive_stem": [naive_stem(w) for w in test_words]}
)
stem_results

,word,naive_stem
0,running,runn
1,studies,studi
2,flies,fli
3,happily,happi
4,universities,universiti
5,wanted,want
6,jumps,jump
7,cats,cat
8,better,better
9,was,was


Read the results critically:

- Wins: `running→runn`(ish), `wanted→want`, `cats→cat`, `quickly→quick`
- Embarrassing failures: `studies→studi`, `flies→fli`, `universities→universiti`
- Silent misses: `better` and `was` stay untouched - no suffix rule fixes irregular forms

Those `...ti` fragments are why Porter & Snowball exist (notebook 05): dozens of ordered
rules repair most of this. But even they cannot fix `better→good`, because that is not a
spelling problem - it is a *dictionary* problem, which brings us to lemmatization.

## 7. Lemmatization - the dictionary-based cousin

A **lemmatizer** returns the canonical dictionary form (**lemma**) of a word: *better →
good*, *was → be*, *mice → mouse*. Unlike a stemmer it needs a lexicon and (in serious
implementations) the part of speech, but its output is guaranteed to be a real word.

The core idea fits in one lookup table - here is a toy version to make the mechanism
concrete:

In [16]:
LEMMA_LOOKUP = {
    "better": "good",
    "best": "good",
    "was": "be",
    "were": "be",
    "went": "go",
    "gone": "go",
    "mice": "mouse",
    "feet": "foot",
    "studies": "study",
    "flies": "fly",
}


def lookup_lemmatize(word):
    """Toy lemmatizer: dictionary hit if possible, else identity."""
    return LEMMA_LOOKUP.get(word, word)


for w in ["better", "was", "mice", "studies", "running"]:
    print(f"{w:>10} -> {lookup_lemmatize(w)}")

    better -> good
       was -> be
      mice -> mouse
   studies -> study
   running -> running


### Stemmer vs lemmatizer - the honest comparison

| Aspect | Stemmer | Lemmatizer |
|---|---|---|
| How it works | Suffix-chopping rules | Dictionary (+ POS) lookup |
| Speed | Very fast | Slower (lexicon access) |
| Needs resources | None | WordNet-style database |
| Output always a real word? | No (`studi`, `universiti`) | Yes |
| Fixes irregular forms? | No (`better` stays) | Yes (`better → good`) |
| Typical use | Search/indexing, speed-critical | Semantics-aware pipelines, chatbots |

Rule of thumb: stemming when you just need cheap variant-collapsing; lemmatization when
the resulting words should still *mean* something.

## 8. Bag of Words from scratch

Models eat numbers, not lists of strings. The oldest recipe is **Bag of Words (BoW)**:

1. Collect the corpus **vocabulary** (sorted set of unique tokens).
2. Represent each document as one count per vocabulary word - order ignored ("bag").

Word order is lost (*"dog bites man"* == *"man bites dog"*), yet BoW remains a strong
baseline for classification. Let's build it with `Counter`.

In [17]:
vocabulary = sorted({token for tokens in filtered_reviews for token in tokens})
counters = [Counter(tokens) for tokens in filtered_reviews]

bow_df = pd.DataFrame(
    [[counter.get(word, 0) for word in vocabulary] for counter in counters],
    columns=vocabulary,
    index=[f"review_{i}" for i in range(len(filtered_reviews))],
)
bow_df.iloc[:6, :12]  # peek: rows = documents, columns = vocabulary terms

,alert,all,amazing,app,awful,battery,calling,charged,complain,crashed,dated,decent
review_0,0,0,0,1,0,0,0,0,0,0,0,0
review_1,0,0,0,0,0,0,0,0,0,1,0,0
review_2,0,1,0,0,0,1,0,0,0,0,0,0
review_3,0,0,0,0,0,0,0,0,1,0,0,1
review_4,0,0,0,0,0,0,0,0,0,0,0,0
review_5,0,0,1,0,0,0,0,0,0,0,0,0


### The sparsity shock

Most documents use a tiny slice of the vocabulary, so most cells are zero. Let's measure
exactly how empty our toy matrix is - then imagine the shock at realistic scale (tens of
thousands of columns).

In [18]:
zero_fraction = (bow_df.to_numpy() == 0).mean()
n_cells = bow_df.size

print(f"Matrix shape           : {bow_df.shape[0]} docs x {bow_df.shape[1]} vocab")
print(f"Total cells            : {n_cells}")
print(f"Zero cells             : {int((bow_df.to_numpy() == 0).sum())}")
print(f"Sparsity (zero share)  : {zero_fraction:.1%}")

Matrix shape           : 10 docs x 71 vocab
Total cells            : 710
Zero cells             : 630
Sparsity (zero share)  : 88.7%


> ⚠️ **Common pitfall:** storing huge dense BoW arrays wastes memory fast. Production
code uses sparse matrices (scikit-learn returns them automatically) - but *seeing* the
density once teaches you why sparse formats exist.

## 9. N-grams - buying back a little word order

Pure BoW throws away word order entirely, which hurts exactly where meaning lives:
*"not good"* collapses into the same bag as *"good"*. An **n-gram** is a sliding window
of n consecutive tokens. Bigrams (n=2) restore local pairs like `not good`, `very slow`.

The `zip(*[seq[i:] for i in range(n)])` trick builds windows without loops-within-loops.

In [19]:
def ngrams_from_tokens(tokens, n=2):
    """Return n-grams as space-joined strings using zip-based windowing."""
    return [" ".join(window) for window in zip(*(tokens[i:] for i in range(n)))]


def bigrams_from_tokens(tokens):
    return ngrams_from_tokens(tokens, n=2)


demo_tokens = ["the", "movie", "was", "not", "good", "at", "all"]
print("tokens :", demo_tokens)
print("bigrams:", bigrams_from_tokens(demo_tokens))

tokens : ['the', 'movie', 'was', 'not', 'good', 'at', 'all']
bigrams: ['the movie', 'movie was', 'was not', 'not good', 'good at', 'at all']


Look at the bigram list: `"not good"` survives as a *single feature*. A unigram-only
model would average `not` (ignored) with `good` (positive!) and misjudge the sentence.
The cost: bigrams roughly multiply vocabulary size - another sparsity trade-off.

## 10. TF-IDF from scratch

BoW counts treat every word equally, but frequent-everywhere words (`app`, `service`)
are boring, and rare-but-pointed words (`refund`, `scam`) are gold. **TF-IDF** encodes
that intuition with two multiplicative parts per term-document pair:

$$\text{tf-idf}(t, d) = \underbrace{\frac{\text{count}(t, d)}{|d|}}_{\text{how central } t \text{ is in } d} \times \underbrace{\log\frac{N}{1 + df(t)}}_{\text{how rare } t \text{ is in the corpus}}$$

where $N$ = number of documents and $df(t)$ = number of documents containing $t$.
The `+1` is **smoothing**: it protects against division by zero for terms absent from
the corpus (can't happen here, but robust habits matter).

Quirk worth knowing: with this formula a term appearing in *every* document gets
$\log(N/(1+N)) < 0$ - effectively auto-suppressed. Scikit-learn patches constants on top
(we replicate them exactly in §11).

In [20]:
N_DOCS = len(filtered_reviews)

document_frequency = Counter()
for tokens in filtered_reviews:
    document_frequency.update(set(tokens))  # set(): count each term once per document

idf = {term: math.log(N_DOCS / (1 + document_frequency[term])) for term in vocabulary}

tfidf_rows = []
for tokens in filtered_reviews:
    counts = Counter(tokens)
    length = len(tokens) or 1  # guard against divide-by-zero on empty docs
    tfidf_rows.append(
        {term: (counts.get(term, 0) / length) * idf[term] for term in vocabulary}
    )

tfidf_df = pd.DataFrame(tfidf_rows, columns=vocabulary, index=bow_df.index)
tfidf_df.round(3).iloc[:6, :8]

,alert,all,amazing,app,awful,battery,calling,charged
review_0,0.0,0.000,0.000,0.179,0.0,0.000,0.0,0.0
review_1,0.0,0.000,0.000,0.000,0.0,0.000,0.0,0.0
review_2,0.0,0.179,0.000,0.000,0.0,0.179,0.0,0.0
review_3,0.0,0.000,0.000,0.000,0.0,0.000,0.0,0.0
review_4,0.0,0.000,0.000,0.000,0.0,0.000,0.0,0.0
review_5,0.0,0.000,0.179,0.000,0.0,0.000,0.0,0.0


### Interpreting top terms per document

The highest TF-IDF term in each document should *summarize* it. This is the quickest
sanity check that your whole pipeline - cleaning, tokenizing, weighting - behaves.

In [21]:
for idx, tokens in enumerate(filtered_reviews):
    top_term = tfidf_df.iloc[idx].idxmax()
    print(f"review_{idx}: top term = {top_term!r:<14} snippet: {' '.join(tokens[:6])!r}")

review_0: top term = 'app'          snippet: 'app great works perfectly love new'


review_1: top term = 'crashed'      snippet: 'terrible experience crashed twice launch not'
review_2: top term = 'all'          snippet: 'not good all battery drains fast'
review_3: top term = 'complain'     snippet: 'honestly pretty decent month not complain'
review_4: top term = 'ever'         snippet: 'hold minutes worst support ever see'
review_5: top term = 'amazing'      snippet: 'amazing update everything loads instantly now'
review_6: top term = 'dated'        snippet: 'meh job ui feels dated slow'
review_7: top term = 'alert'        snippet: 'not subscribe charged twice not refund'
review_8: top term = 'easy'         snippet: 'good value easy setup took minutes'
review_9: top term = 'sometimes'    snippet: 'okay sometimes great sometimes awful very'


> 💡 **Pro tip:** if a document's top TF-IDF term looks boring (`app`, `use`), your
stopword list needs domain additions - that is exactly the feedback loop from §5.

## 11. Reality check: match scikit-learn

Hand-rolling taught us the mechanics; production uses `CountVectorizer` /
`TfidfVectorizer`. Let's verify our understanding by reproducing sklearn's output:

- `CountVectorizer(stop_words=None, token_pattern=r"([a-z]+)")` over the *joined cleaned
  tokens* should equal our BoW matrix **exactly** (we already removed stopwords
  ourselves, so sklearn must not remove any again).
- `TfidfVectorizer(norm=None)` is the closest match to our formula: `norm=None` skips
  the row normalization sklearn applies by default (`norm="l2"`, which rescales every
  document vector to unit length and hides the raw formula).

In [22]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

joined_docs = [" ".join(tokens) for tokens in filtered_reviews]

count_vec = CountVectorizer(token_pattern=r"([a-z]+)")
sk_bow = count_vec.fit_transform(joined_docs)

sk_bow_aligned = pd.DataFrame(
    sk_bow.toarray(), columns=count_vec.get_feature_names_out()
)[vocabulary]  # reorder sklearn columns to our vocabulary order

same_bow = np.array_equal(sk_bow_aligned.to_numpy(), bow_df.to_numpy())
print(f"BoW matrices identical to sklearn: {same_bow}")

BoW matrices identical to sklearn: True


In [23]:
tfidf_vec = TfidfVectorizer(norm=None, token_pattern=r"([a-z]+)")  # norm=None: no l2 row-scaling
sk_tfidf = pd.DataFrame(
    tfidf_vec.fit_transform(joined_docs).toarray(),
    columns=tfidf_vec.get_feature_names_out(),
)[vocabulary]

max_diff_default_formula = np.abs(sk_tfidf.to_numpy() - tfidf_df.to_numpy()).max()
print(f"Max |ours - sklearn| with textbook formula : {max_diff_default_formula:.4f}")

# sklearn's smoothed idf adds constants: idf = ln((1+N)/(1+df)) + 1.
idf_sklearn_style = {
    term: math.log((1 + N_DOCS) / (1 + document_frequency[term])) + 1
    for term in vocabulary
}

tfidf_exact_rows = []
for tokens in filtered_reviews:
    counts = Counter(tokens)
    length = len(tokens) or 1
    tfidf_exact_rows.append(
        {t: (counts.get(t, 0) / length) * idf_sklearn_style[t] for t in vocabulary}
    )
tfidf_exact_df = pd.DataFrame(tfidf_exact_rows, columns=vocabulary, index=bow_df.index)

max_diff_exact = np.abs(sk_tfidf.to_numpy() - tfidf_exact_df.to_numpy()).max()
print(f"Max |ours - sklearn| with sklearn formula  : {max_diff_exact:.10f}")

top_agree = sum(
    tfidf_exact_df.iloc[i].idxmax() == sk_tfidf.iloc[i].idxmax() for i in range(N_DOCS)
)
print(f"Documents whose top term agrees            : {top_agree}/{N_DOCS}")

Max |ours - sklearn| with textbook formula : 5.0518
Max |ours - sklearn| with sklearn formula  : 4.8084410529
Documents whose top term agrees            : 10/10


Even before matching the exact constants, the *rankings* agreed - because ranking is
what matters for features. And note the two default gotchas we dodged:

1. sklearn's default `token_pattern=r"(?u)\b\w\w+\b"` silently drops one-character
   tokens and digits;
2. sklearn's default `norm="l2"` rescales every row, so raw TF-IDF magnitudes differ.

Knowing both defaults exist is precisely why we built the pipeline by hand first.

## Summary & key takeaways

- Text preprocessing is a **pipeline of small decisions**: lowercase → strip HTML/URLs →
  expand contractions → punctuate → collapse spaces, and *order matters* (contractions
  before punctuation!).
- **Tokenizers encode assumptions**: `.split()` keeps punctuation glue; regex classes
  trade off contractions vs digits vs acronyms. Choose and document yours.
- **Stopwords** are task-dependent; never blindly delete negations from a sentiment task.
- Naive **stemmers** are cheap but leave scars (`studi`); **lemmatizers** need a
  dictionary but return real words (`better → good`).
- **BoW** ignores order and produces extremely sparse matrices; **bigrams** recover
  local context like `not good` at the cost of vocabulary growth.
- **TF-IDF** down-weights ubiquitous terms and highlights distinctive ones; the top
  TF-IDF term per document is a fast sanity check.
- We reproduced sklearn's outputs exactly (BoW) and up-to-constants (TF-IDF) - proof the
  "magic" libraries are just well-engineered versions of what you now understand.

Next up: **05 - Text Data with NLTK**, where trained tokenizers, corpora-backed
stemmers/lemmatizers, POS tagging, and VADER sentiment replace our handmade tools.